# 05 - The agents: forecast, uncertainty, critic, planner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/demand-forecasting-capstone/blob/main/notebooks/05_agents_and_modelling.ipynb)

**Goal:** run the multi-agent system and read what each agent decided and why.



How to run: `Runtime -> Run all` (Colab) or run the cells from top to bottom. The default dataset is `synthetic`, which needs no downloads.

In [ ]:
# ---- Setup: works in Google Colab and on your own computer. Nothing to upload. ----
REPO_URL = "https://github.com/YOUR-USERNAME/demand-forecasting-capstone.git"   # <- set once, the same in every notebook
BRANCH = "main"
USE_DRIVE = False   # True = use a copy of the project folder in your Google Drive instead of GitHub

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("src/demand_agents"):
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.chdir("/content/drive/MyDrive/demand-forecasting-capstone")
    elif IN_COLAB:
        if not os.path.exists("/content/project"):
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/project"], check=True)
        os.chdir("/content/project")
    elif os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                                   # running locally from the notebooks folder
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
sys.path.insert(0, os.path.abspath("src"))
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
from IPython.display import display, Image, Markdown
print("Project folder:", os.getcwd())

In [ ]:
DATASET = "synthetic"    # "synthetic" | "synthetic_calm" | "m5" | "m5_christmas" | "walmart" | "walmart_synthetic"
from demand_agents.config import load_named
cfg = load_named(DATASET)
print("dataset:", DATASET, "| source:", cfg["data"]["source"], "| frequency:", cfg["data"]["frequency"])

## The four agents

```
 Data -> Forecaster -> Uncertainty <-> Critic -> Planner -> orders
```

| Agent | Job |
|---|---|
| **Forecaster** | predicts demand `horizon` periods ahead; chooses (on held-out validation data) between a gradient-boosting model, a blend with the classical ETS method, or seasonal naive |
| **Uncertainty** | turns each forecast into a range (90% interval) and a typical error size; checked by the Critic, who can ask for wider ranges |
| **Critic** | audits the ranges and flags risky days (events, abnormal weather, recent big errors, unusual prices); learns how much each warning matters from validation data |
| **Planner** | decides how much stock to hold: expected demand over the protection period plus a safety cushion; tests the policy in a simulation |

Every thought, action and decision is logged (the *reasoning trace*).

## Run the pipeline

Takes about 30 seconds on synthetic data and 5-15 minutes on the real M5 pilot (the classical ETS baseline is the slow part).

In [ ]:
from demand_agents.orchestrator import run_pipeline
state = run_pipeline(cfg)

## What the Forecaster chose

In [ ]:
info = state.metrics["forecaster"]
print("chosen:", info["choice"], "| blend weight on the model:", info["blend_weight"])
display(pd.Series(info["scores"], name="validation WAPE (lower is better)").to_frame())

## What the Critic learned

The hand-set `prior` is the starting guess. `observed` is the extra error seen on validation days with that warning sign. `weight` is the result. A warning sign that does not predict bigger errors is pushed towards zero.

In [ ]:
display(pd.DataFrame(state.metrics.get("risk_weights", {})).T)

## Reasoning traces

In [ ]:
tr = state.tracer.to_dataframe()
display(tr[["step", "agent", "kind", "content"]].head(45))

## Forecast explorer

In [ ]:
import matplotlib.pyplot as plt
fcast = pd.read_csv(f"{cfg['output_dir']}/forecasts.csv", parse_dates=["date"])
def draw(sid):
    s = fcast[(fcast["series_id"] == sid) & fcast["split"].isin(["val_check", "test"])]
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.fill_between(s["date"], s["lower"], s["upper"], alpha=0.2, label="90% range")
    ax.plot(s["date"], s["sales"], "k.-", lw=0.8, label="actual"); ax.plot(s["date"], s["y_hat"], label="forecast")
    fl = s[s["risk_multiplier"] > 1.15]; ax.scatter(fl["date"], fl["sales"], facecolors="none", edgecolors="crimson", marker="D", label="Critic flag")
    ax.set_title(sid); ax.legend(ncol=4, fontsize=8); plt.show()
try:
    import ipywidgets as w
    dd = w.Dropdown(options=sorted(fcast["series_id"].unique()), description="Series"); w.interact(draw, sid=dd)
except Exception:
    draw(sorted(fcast["series_id"].unique())[0])

**Next:** `06_evaluation_and_report.ipynb`.